# MarsLandformNet V2 — SSL LoRA Training
## Instructions:
1. Upload `mars_tiles.tar.gz` to Google Drive root
2. Run all cells (Runtime → Run all)
3. Download `lora_weights/` from Drive when done (~2MB)
4. Training takes ~2-3 hours on T4 GPU


In [ ]:
!pip install -q peft transformers torch torchvision tqdm


In [ ]:
from pathlib import Path
import tarfile

from google.colab import drive

drive.mount('/content/drive')

tar_path = Path('/content/drive/MyDrive/mars_tiles.tar.gz')
extract_root = Path('/content')
if not tar_path.exists():
    raise FileNotFoundError(f"Could not find {tar_path}. Upload mars_tiles.tar.gz to Drive root.")

print(f"Extracting {tar_path} to {extract_root} ...")
with tarfile.open(tar_path, 'r:gz') as tf:
    tf.extractall(path=extract_root)

expected_data_dir = Path('/content/tiles')
if not expected_data_dir.exists():
    raise FileNotFoundError("Expected extracted data at /content/tiles. Check archive structure.")

patterns = ('*.jpg', '*.jpeg', '*.JPG', '*.JPEG')
all_tiles = []
for pattern in patterns:
    all_tiles.extend(expected_data_dir.rglob(pattern))

tile_count = len(all_tiles)
print(f"Found {tile_count:,} tiles under {expected_data_dir}")
if tile_count <= 0:
    raise RuntimeError("No tiles found after extraction. Archive may be empty or malformed.")


In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Tuple

import torch

@dataclass
class SSLConfig:
    # Model
    model_name: str = 'facebook/dinov2-base'
    embed_dim: int = 768
    image_size: int = 224

    # LoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1
    lora_target_modules: List[str] = field(default_factory=lambda: ['query', 'key', 'value'])
    unfreeze_last_n_blocks: int = 2

    # SSL training
    ssl_lr: float = 5e-5
    ssl_layerwise_decay: float = 0.9
    ssl_epochs: int = 50
    ssl_warmup_epochs: int = 5
    ssl_batch_size: int = 16
    ssl_temperature: float = 0.04
    ssl_crop_scale: Tuple[float, float] = (0.4, 1.0)

    # Augmentations (NO color jitter)
    aug_rotation_degrees: List[int] = field(default_factory=lambda: [0, 90, 180, 270])
    aug_hflip: bool = True
    aug_vflip: bool = True
    aug_gaussian_noise_std: float = 0.02
    aug_use_color_jitter: bool = False

    # Paths / runtime
    data_dir: Path = Path('/content/tiles')
    output_dir: Path = Path('/content/lora_weights')
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    mixed_precision: bool = torch.cuda.is_available()
    num_workers: int = 0

config = SSLConfig()
data_dir = config.data_dir
output_dir = config.output_dir
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Device: {config.device}")
if config.device != 'cuda':
    print('WARNING: GPU not detected. Training will be much slower on CPU.')
print(f"Batch size: {config.ssl_batch_size} | Epochs: {config.ssl_epochs}")
print(f"Data dir: {data_dir}")
print(f"Output dir: {output_dir}")


In [ ]:
from pathlib import Path
from typing import Any, Optional

import torch
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from torch import Tensor, nn
from transformers import Dinov2Model

class DinoV2LoRA(nn.Module):
    def __init__(
        self,
        config: SSLConfig,
        use_lora: bool = True,
        lora_weights_path: Optional[str] = None,
    ) -> None:
        super().__init__()
        self.config = config
        self.use_lora = use_lora

        try:
            self.backbone = Dinov2Model.from_pretrained(config.model_name, low_cpu_mem_usage=True)
        except Exception as exc:
            raise RuntimeError(f"Failed to load DINOv2 backbone '{config.model_name}': {exc}") from exc

        if self.use_lora:
            lora_cfg = LoraConfig(
                r=config.lora_r,
                lora_alpha=config.lora_alpha,
                lora_dropout=config.lora_dropout,
                target_modules=config.lora_target_modules,
                task_type=TaskType.FEATURE_EXTRACTION,
                bias='none',
            )
            self.backbone = get_peft_model(self.backbone, lora_cfg)
            if lora_weights_path is not None:
                self.load_pretrained(lora_weights_path)

        self._freeze_all_parameters()
        self._unfreeze_last_blocks(config.unfreeze_last_n_blocks)
        self._print_trainable_params()

    def _freeze_all_parameters(self) -> None:
        for param in self.backbone.parameters():
            param.requires_grad = False

        if self.use_lora:
            for name, param in self.backbone.named_parameters():
                if 'lora_' in name:
                    param.requires_grad = True

    def _unfreeze_last_blocks(self, n_blocks: int) -> None:
        if n_blocks <= 0:
            return

        blocks = self._encoder_layers()
        n_blocks = min(n_blocks, len(blocks))
        for block in blocks[-n_blocks:]:
            for param in block.parameters():
                param.requires_grad = True

    def _encoder_layers(self) -> list[nn.Module]:
        if self.use_lora:
            base_model = getattr(self.backbone, 'base_model', None)
            model = getattr(base_model, 'model', None)
        else:
            model = self.backbone

        encoder = getattr(model, 'encoder', None)
        layers = getattr(encoder, 'layer', None)
        if not isinstance(layers, nn.ModuleList):
            raise RuntimeError('Could not resolve DINOv2 encoder layers.')
        return list(layers)

    def num_layers(self) -> int:
        return len(self._encoder_layers())

    def _print_trainable_params(self) -> None:
        total = sum(p.numel() for p in self.backbone.parameters())
        trainable = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        pct = (100.0 * trainable / total) if total > 0 else 0.0
        print(f"[DinoV2LoRA] trainable params: {trainable:,}/{total:,} ({pct:.2f}%)")

    def forward(self, pixel_values: Tensor) -> Tensor:
        outputs = self.backbone(pixel_values=pixel_values)
        cls = outputs.last_hidden_state[:, 0]
        if cls.shape[-1] != self.config.embed_dim:
            raise RuntimeError(
                f"Unexpected CLS embedding dim {cls.shape[-1]} (expected {self.config.embed_dim})"
            )
        return cls

    def save_pretrained(self, path: str | Path) -> None:
        output_path = Path(path)
        output_path.mkdir(parents=True, exist_ok=True)
        self.backbone.save_pretrained(str(output_path))

    def load_pretrained(self, path: str | Path) -> None:
        model_path = Path(path)
        if not model_path.exists():
            raise FileNotFoundError(f"LoRA checkpoint path does not exist: {model_path}")
        if not self.use_lora:
            raise RuntimeError('load_pretrained is only supported when use_lora=True')
        self.backbone = PeftModel.from_pretrained(self.backbone, str(model_path), is_trainable=True)


In [ ]:
from collections.abc import Sequence
from typing import cast

import torch
from PIL import Image
from torch import Tensor
from torchvision import transforms

class RandomDiscreteRotation:
    def __init__(self, angles: Sequence[int]) -> None:
        if not angles:
            raise ValueError('angles must be non-empty')
        self.angles = list(angles)

    def __call__(self, img: Image.Image) -> Image.Image:
        idx = int(torch.randint(0, len(self.angles), (1,)).item())
        return img.rotate(int(self.angles[idx]))

class GaussianNoise:
    def __init__(self, std: float = 0.02) -> None:
        self.std = std

    def __call__(self, x: Tensor) -> Tensor:
        if self.std <= 0:
            return x
        return x + torch.randn_like(x) * self.std

class MarsDINOCrops:
    def __init__(self, cfg: SSLConfig) -> None:
        if cfg.aug_use_color_jitter:
            raise ValueError('Color jitter must remain disabled for Mars SSL training.')

        normalize = transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        )

        common = [
            RandomDiscreteRotation(cfg.aug_rotation_degrees),
            transforms.RandomHorizontalFlip(p=0.5 if cfg.aug_hflip else 0.0),
            transforms.RandomVerticalFlip(p=0.5 if cfg.aug_vflip else 0.0),
            transforms.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0)),
            transforms.ToTensor(),
            GaussianNoise(std=cfg.aug_gaussian_noise_std),
            normalize,
        ]

        self.global_transform = transforms.Compose([
            transforms.RandomResizedCrop(
                size=cfg.image_size,
                scale=tuple(cfg.ssl_crop_scale),
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            *common,
        ])

        self.local_transform = transforms.Compose([
            transforms.RandomResizedCrop(
                size=cfg.image_size,
                scale=(0.05, 0.4),
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            *common,
        ])

        self.num_local_crops = 4

    def __call__(self, img: Image.Image) -> list[Tensor]:
        crops = [
            cast(Tensor, self.global_transform(img)),
            cast(Tensor, self.global_transform(img)),
        ]
        for _ in range(self.num_local_crops):
            crops.append(cast(Tensor, self.local_transform(img)))
        return crops


In [ ]:
from pathlib import Path

import torch
from PIL import Image
from torch import Tensor
from torch.utils.data import DataLoader, Dataset

class MarsImageDataset(Dataset[list[Tensor]]):
    def __init__(self, data_dir: Path, transform: MarsDINOCrops) -> None:
        self.data_dir = data_dir
        self.transform = transform

        patterns = ('*.jpg', '*.jpeg', '*.JPG', '*.JPEG')
        self.image_paths = []
        for pattern in patterns:
            self.image_paths.extend(sorted(data_dir.rglob(pattern)))

        if not self.image_paths:
            raise FileNotFoundError(f'No JPEG images found under: {data_dir}')

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, index: int) -> list[Tensor]:
        img_path = self.image_paths[index]
        try:
            with Image.open(img_path) as img:
                img = img.convert('RGB')
                return self.transform(img)
        except Exception as exc:
            raise RuntimeError(f'Failed loading image {img_path}: {exc}') from exc

def dino_collate_fn(batch: list[list[Tensor]]) -> list[Tensor]:
    n_crops = len(batch[0])
    collated = []
    for idx in range(n_crops):
        collated.append(torch.stack([sample[idx] for sample in batch], dim=0))
    return collated

transform = MarsDINOCrops(config)
dataset = MarsImageDataset(data_dir=data_dir, transform=transform)
loader = DataLoader(
    dataset,
    batch_size=config.ssl_batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
    collate_fn=dino_collate_fn,
)

print(f'Dataset size: {len(dataset):,} images')
print(f'Steps per epoch: {len(loader):,}')
if len(loader) == 0:
    raise RuntimeError('No training steps available. Reduce batch size or add more data.')


In [ ]:
import copy
import gc
import math
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from tqdm.auto import tqdm

class DINOHead(nn.Module):
    def __init__(self, in_dim: int, out_dim: int = 4096) -> None:
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.GELU(),
            nn.Linear(in_dim, in_dim),
            nn.GELU(),
            nn.Linear(in_dim, out_dim),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.mlp(x)
        return F.normalize(x, dim=-1)

class DINOTrainer:
    def __init__(self, cfg: SSLConfig, data_dir: Path, output_dir: Path) -> None:
        self.cfg = cfg
        self.epochs = cfg.ssl_epochs
        self.output_dir = output_dir
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.device = torch.device(cfg.device)
        self.mixed_precision = cfg.mixed_precision

        transform = MarsDINOCrops(cfg)
        dataset = MarsImageDataset(data_dir=data_dir, transform=transform)
        self.loader = DataLoader(
            dataset,
            batch_size=cfg.ssl_batch_size,
            shuffle=True,
            num_workers=cfg.num_workers,
            pin_memory=False,
            drop_last=True,
            collate_fn=dino_collate_fn,
        )
        if len(self.loader) == 0:
            raise RuntimeError('No training steps available. Reduce batch size or add more data.')

        # Build student backbone with LoRA
        self.student_backbone = DinoV2LoRA(cfg, use_lora=True).to(self.device)
        gc.collect()
        torch.cuda.empty_cache()

        # Build teacher WITHOUT deepcopy to save RAM
        self.teacher_backbone = DinoV2LoRA(cfg, use_lora=True).to(self.device)
        self.teacher_backbone.load_state_dict(self.student_backbone.state_dict())
        gc.collect()
        torch.cuda.empty_cache()

        self.student_head = DINOHead(cfg.embed_dim).to(self.device)
        self.teacher_head = DINOHead(cfg.embed_dim).to(self.device)
        self.teacher_head.load_state_dict(self.student_head.state_dict())

        for module in (self.teacher_backbone, self.teacher_head):
            module.eval()
            for p in module.parameters():
                p.requires_grad = False

        self.optimizer = AdamW(self._build_param_groups(), lr=cfg.ssl_lr, weight_decay=0.04)

        total_steps = self.epochs * len(self.loader)
        warmup_steps = max(1, cfg.ssl_warmup_epochs * len(self.loader))

        def lr_lambda(step: int) -> float:
            if step < warmup_steps:
                return float(step + 1) / float(warmup_steps)
            progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
            return 0.5 * (1.0 + math.cos(math.pi * progress))

        self.scheduler = LambdaLR(self.optimizer, lr_lambda=lr_lambda)
        self.scaler = torch.amp.GradScaler('cuda', enabled=self.mixed_precision)

        self.student_temp = 0.1
        self.teacher_temp = cfg.ssl_temperature
        self.center_momentum = 0.9
        self.registered_center = torch.zeros(1, 4096, device=self.device)

        self.best_loss = float('inf')
        self.loss_history = []
        self.lr_history = []
        self.final_loss = None

    def _build_param_groups(self) -> list[dict[str, object]]:
        groups = []
        layer_count = self.student_backbone.num_layers()

        def layer_scale(layer_idx: int) -> float:
            depth = (layer_count - 1) - layer_idx
            return self.cfg.ssl_layerwise_decay ** depth

        for name, param in self.student_backbone.named_parameters():
            if not param.requires_grad:
                continue

            lr = self.cfg.ssl_lr
            if 'encoder.layer.' in name:
                layer_idx = int(name.split('encoder.layer.')[1].split('.')[0])
                lr = self.cfg.ssl_lr * layer_scale(layer_idx)

            groups.append({
                'params': [param],
                'lr': lr,
                'weight_decay': 0.0 if name.endswith('bias') else 0.04,
            })

        groups.append({
            'params': list(self.student_head.parameters()),
            'lr': self.cfg.ssl_lr,
            'weight_decay': 0.04,
        })
        return groups

    @torch.no_grad()
    def _update_teacher(self, momentum: float) -> None:
        for student_p, teacher_p in zip(self.student_backbone.parameters(), self.teacher_backbone.parameters()):
            teacher_p.data.mul_(momentum).add_(student_p.data * (1.0 - momentum))
        for student_p, teacher_p in zip(self.student_head.parameters(), self.teacher_head.parameters()):
            teacher_p.data.mul_(momentum).add_(student_p.data * (1.0 - momentum))

    @torch.no_grad()
    def _update_center(self, teacher_logits: Tensor) -> None:
        batch_center = torch.mean(teacher_logits, dim=0, keepdim=True)
        self.registered_center = (
            self.registered_center * self.center_momentum + batch_center * (1.0 - self.center_momentum)
        )

    def _dino_loss(self, student_out: list[Tensor], teacher_out: list[Tensor]) -> Tensor:
        teacher_probs = [
            F.softmax((x - self.registered_center) / self.teacher_temp, dim=-1).detach()
            for x in teacher_out
        ]
        student_log_probs = [F.log_softmax(x / self.student_temp, dim=-1) for x in student_out]

        total_loss = torch.tensor(0.0, device=self.device)
        n_terms = 0
        for t_idx, t_prob in enumerate(teacher_probs):
            for s_idx, s_log_prob in enumerate(student_log_probs):
                if s_idx == t_idx:
                    continue
                total_loss += torch.mean(torch.sum(-t_prob * s_log_prob, dim=-1))
                n_terms += 1

        if n_terms == 0:
            raise RuntimeError('No DINO loss terms were computed.')
        return total_loss / n_terms

    def _momentum_at_step(self, step: int, total_steps: int) -> float:
        base = 0.996
        cosine = 0.5 * (1.0 + math.cos(math.pi * step / max(1, total_steps)))
        return 1.0 - (1.0 - base) * cosine

    def _save_checkpoint(self, path: Path, epoch: int, loss_value: float) -> None:
        state = {
            'epoch': epoch,
            'loss': loss_value,
            'student_backbone': self.student_backbone.state_dict(),
            'student_head': self.student_head.state_dict(),
            'teacher_backbone': self.teacher_backbone.state_dict(),
            'teacher_head': self.teacher_head.state_dict(),
            'optimizer': self.optimizer.state_dict(),
            'scheduler': self.scheduler.state_dict(),
            'center': self.registered_center,
        }
        torch.save(state, path)

    def train(self) -> None:
        global_step = 0
        total_steps = self.epochs * len(self.loader)
        momentum = 0.996
        accum_steps = 4  # gradient accumulation to simulate batch_size=64

        # Enable gradient checkpointing to save VRAM
        if hasattr(self.student_backbone, 'gradient_checkpointing_enable'):
            self.student_backbone.gradient_checkpointing_enable()

        for epoch in range(1, self.epochs + 1):
            self.student_backbone.train()
            self.student_head.train()
            running_loss = 0.0

            for batch_idx, crops in enumerate(tqdm(self.loader, desc=f'Epoch {epoch}/{self.epochs}', leave=False)):
                crops = [c.to(self.device, non_blocking=True) for c in crops]
                teacher_views = crops[:2]

                momentum = self._momentum_at_step(global_step, total_steps)

                with torch.amp.autocast(device_type=self.device.type, enabled=self.mixed_precision):
                    student_embeddings = [self.student_backbone(view) for view in crops]
                    student_logits = [self.student_head(x) for x in student_embeddings]

                    with torch.no_grad():
                        teacher_embeddings = [self.teacher_backbone(view) for view in teacher_views]
                        teacher_logits = [self.teacher_head(x) for x in teacher_embeddings]

                    loss = self._dino_loss(student_logits, teacher_logits) / accum_steps

                self.scaler.scale(loss).backward()

                if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(self.loader):
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.optimizer.zero_grad(set_to_none=True)
                    self.scheduler.step()

                with torch.no_grad():
                    cat_teacher = torch.cat(teacher_logits, dim=0)
                    self._update_center(cat_teacher)
                    self._update_teacher(momentum)

                running_loss += float(loss.item()) * accum_steps

                # Free memory
                del crops, student_embeddings, student_logits, teacher_embeddings, teacher_logits, loss
                if batch_idx % 20 == 0:
                    torch.cuda.empty_cache()

                global_step += 1

            epoch_loss = running_loss / len(self.loader)
            current_lr = float(self.optimizer.param_groups[0]['lr'])
            self.loss_history.append(epoch_loss)
            self.lr_history.append(current_lr)
            self.final_loss = epoch_loss

            print(
                f'Epoch {epoch:03d}/{self.epochs} | '
                f'loss={epoch_loss:.5f} | lr={current_lr:.6e} | ema_m={momentum:.6f}'
            )

            if epoch % 5 == 0:
                self._save_checkpoint(self.output_dir / f'checkpoint_epoch_{epoch}.pt', epoch, epoch_loss)

            if epoch_loss < self.best_loss:
                self.best_loss = epoch_loss
                self._save_checkpoint(self.output_dir / 'best_model.pt', epoch, epoch_loss)


In [ ]:
def count_trainable_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

trainer = DINOTrainer(config, data_dir, output_dir)
trainable_backbone = count_trainable_params(trainer.student_backbone)
trainable_head = count_trainable_params(trainer.student_head)
print(f'Trainable backbone params: {trainable_backbone:,}')
print(f'Trainable head params: {trainable_head:,}')
print(f'Total trainable params: {trainable_backbone + trainable_head:,}')

trainer.train()
print(f'SSL Training Complete! Final loss: {trainer.final_loss:.5f}')


In [ ]:
# Save just the LoRA adapter weights (~2MB) to Drive
save_dir = '/content/drive/MyDrive/marslandform_lora_weights'
trainer.student_backbone.save_pretrained(save_dir)
print('LoRA weights saved to Google Drive!')


In [ ]:
# OPTIONAL: quality check with t-SNE before/after (can be skipped)
import random

import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader
from torchvision import transforms

optional_sample_size = 1000
all_paths = dataset.image_paths
if len(all_paths) < optional_sample_size:
    optional_sample_size = len(all_paths)

sample_paths = random.sample(all_paths, optional_sample_size)

embed_transform = transforms.Compose([
    transforms.Resize((config.image_size, config.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

class EmbeddingDataset(torch.utils.data.Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        with Image.open(p) as img:
            return self.transform(img.convert('RGB'))

embed_ds = EmbeddingDataset(sample_paths, embed_transform)
embed_loader = DataLoader(embed_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

def extract_embeddings(model_backbone):
    model_backbone.eval()
    embs = []
    with torch.no_grad():
        for batch in embed_loader:
            batch = batch.to(trainer.device, non_blocking=True)
            embs.append(model_backbone(batch).cpu())
    return torch.cat(embs, dim=0).numpy()

frozen_backbone = DinoV2LoRA(config, use_lora=False).to(trainer.device)
base_emb = extract_embeddings(frozen_backbone)
lora_emb = extract_embeddings(trainer.student_backbone)

reducer = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
base_2d = reducer.fit_transform(base_emb)
lora_2d = reducer.fit_transform(lora_emb)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(base_2d[:, 0], base_2d[:, 1], s=5, alpha=0.6)
axes[0].set_title('Frozen DINOv2 Embeddings (t-SNE)')
axes[0].set_xticks([])
axes[0].set_yticks([])

axes[1].scatter(lora_2d[:, 0], lora_2d[:, 1], s=5, alpha=0.6, c='tab:orange')
axes[1].set_title('LoRA-Finetuned Embeddings (t-SNE)')
axes[1].set_xticks([])
axes[1].set_yticks([])

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

if len(getattr(trainer, 'loss_history', [])) == 0:
    raise RuntimeError('No training history found. Run training cell first.')

epochs = list(range(1, len(trainer.loss_history) + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, trainer.loss_history, marker='o', linewidth=1.5)
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)

if len(getattr(trainer, 'lr_history', [])) > 0:
    axes[1].plot(epochs, trainer.lr_history, marker='o', linewidth=1.5, color='tab:green')
    axes[1].set_title('Learning Rate Schedule')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('LR')
    axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'LR history unavailable', ha='center', va='center')
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()
